In [1]:
import pandas as pd
import altair as alt
import geopandas as gpd

The data set we will be using is about trees in Seattle, from the Seattle Open Data. Details on the data set can be found here: https://www.seattle.gov/Documents/Departments/SDOT/GIS/Trees_OD.pdf

We will also be importing a map of Seattle specifically (https://github.com/seattleflu/seattle-geojson/tree/master/seattle_geojsons)
I have included a code snippet to use if geopandas does not work for you. Otherwise, use the last line to import your data. 

In [28]:
trees = pd.read_csv('small_trees.csv', index_col=0)
trees.head()

,X,Y,PRIMARYDISTRICTCD,COMMON_NAME,TREEHEIGHT,DIAM,GENUS,HERITAGE,EXCEPTIONAL
0,-122.337739,47.626592,DISTRICT3,Laceleaf Maple,6.0,7,Acer,Y,N
1,-122.399420,47.659269,DISTRICT7,Coast Redwood,99.0,52,Sequoia,Y,N
2,-122.314508,47.625651,DISTRICT3,Sweetgum,0.0,30,Liquidambar,Y,N
3,-122.338973,47.664247,DISTRICT4,European Chestnut,36.0,55,Castanea,Y,Y
4,-122.314534,47.625649,DISTRICT3,"Tupelo, Sour Gum",0.0,28,Nyssa,Y,N


In [56]:
# use this only if geopandas does not work for you
def fix_geojson(data):
    props = [val['properties'] for val in data['features']]
    df = pd.DataFrame.from_dict(props)
    
    # Safely handle 'type' in properties if it exists
    if 'type' in df.columns:
        df['proptype'] = df['type']
    df['type'] = 'Feature'  # this is the GeoJSON feature type
    
    geoms = [val['geometry'] for val in data['features']]    
    df['geometry'] = geoms
    
    # Handle Point geometries
    if geoms[0]['type'] == 'Point':
        df['lon'] = [geom['coordinates'][0] for geom in geoms]
        df['lat'] = [geom['coordinates'][1] for geom in geoms]
    
    dalt = alt.Data(values=df.to_dict(orient='records'))
    return dalt

#seattle = fix_geojson(pd.read_json('2016_seattle_neighborhoods.geojson'))
seattle = gpd.read_file('2016_seattle_neighborhoods.geojson', driver = 'GeoJSON')
seattle.head()

/opt/anaconda3/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(


,STATEFP,COUNTYF,TRACTCE,AFFGEOI,GEOID,NAME,LSAD,ALAND,AWATER,rowID,CRA_NAM,NEIGHBO,PUMA5CE,geometry
0,1400000US53033003000,033,003000,1400000US53033003000,53033003000,30,CT,1493249,0,1,Whittier Heights,Ballard,11601,"MULTIPOLYGON (((-122.36617 47.67235, -122.3661..."
1,1400000US53033005801,033,005801,1400000US53033005801,53033005801,58.01,CT,1814668,264894,2,Interbay,Magnolia/Queen Anne,11603,"MULTIPOLYGON (((-122.35149 47.62457, -122.3476..."
2,1400000US53033007600,033,007600,1400000US53033007600,53033007600,76,CT,571879,0,3,Miller Park,East,11604,"MULTIPOLYGON (((-122.28504 47.64752, -122.2569..."
3,1400000US53033000200,033,000200,1400000US53033000200,53033000200,2,CT,3286278,0,4,Olympic Hills/Victory Heights,North,11602,"MULTIPOLYGON (((-122.29088 47.70478, -122.2907..."
4,1400000US53033009300,033,009300,1400000US53033009300,53033009300,93,CT,9429073,719985,5,Duwamish/SODO,Greater Duwamish,11605,"MULTIPOLYGON (((-122.2783 47.53154, -122.27808..."


### Basic Map

Make a map of Seattle using the `seattle` GeoJSON file, saving it as a base map. Then, layer on a scatterplot of trees. Choose a projection and size that makes sense for the shape/orientation of Seattle. 

In [108]:

# Base map: polygons only
basemap = (
    alt.Chart(alt.Data(values=seattle_json['features']))
    .mark_geoshape(
        fill='white',
        stroke='black',
        strokeWidth=0.5
    )
    .properties(
        width=500,
        height=800,
        title='Seattle Trees Density'
    )
    .project(type='mercator')
)


#Tree scatter layer
tree_points = alt.Chart(trees).mark_circle(
    opacity=0.7,
    color='green'
).encode(
    longitude='X:Q',
    latitude='Y:Q',
    tooltip=[
        'COMMON_NAME:N',
        'GENUS:N',
        'DIAM:Q',
        'TREEHEIGHT:Q',
        'PRIMARYDISTRICTCD:N'
    ]
)
map_with_trees = basemap + tree_points
map_with_trees

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

alt.LayerChart(...)

### Interactive Altair Map

Using your first map as a base, make a new one with interaction. 

First, (separately), create a histogram of tree diameters. Bin, scale, or edit it in some way so that the entire distribution can be seen.

Now, add a brush selection to the histogram to select specific diameters of trees, and change the opacity of the mapped points based on your selection.

In [110]:
brush = alt.selection_interval(encodings=['x'])

hist_brushed = (
    alt.Chart(trees)
    .mark_bar()
    .encode(
        x=alt.X(
            'DIAM:Q',
            bin=alt.Bin(maxbins=100),
            title='Tree Diameter'
        ),
        y=alt.Y('count():Q', title='Count of Trees')
    )
    .properties(
        width=500,
        height=150,
        title='Filter Trees by Diameter'
    )
    .add_params(brush)
)

tree_points_brushed = tree_points.encode(
    opacity=alt.condition(brush, alt.value(0.9), alt.value(0.1))
)

map_with_trees_interactive = basemap + tree_points_brushed

final_chart = map_with_trees_interactive | hist_brushed
final_chart


/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

alt.HConcatChart(...)